## Preparing the Dataset

In [151]:
# checking venv

import sys
print(sys.executable)

/home/gusbeihel@GU.GU.SE/ml2-final-project/.venv/bin/python


In [152]:
import json
import pandas as pd
import numpy as np

In [153]:
# loading the original data (not clean JSONL)

def load_json_objects(filepath):
    with open(filepath, encoding="utf-8") as f:
        content = f.read()

    decoder = json.JSONDecoder()
    records = []
    position = 0

    while position < len(content):
        # Skip whitespace between objects
        while position < len(content) and content[position].isspace():
            position += 1

        if position >= len(content):
            break

        obj, position = decoder.raw_decode(content, position)
        records.append(obj)

    return pd.DataFrame(records)

train_split = load_json_objects("../data/train.jsonl")
dev_split = load_json_objects("../data/dev.jsonl")
test_split = load_json_objects("../data/test.jsonl")

print("Train:", train_split.shape)
print("Dev:", dev_split.shape)
print("Test:", test_split.shape)

display(train_split.head())

Train: (384, 11)
Dev: (103, 11)
Test: (484, 11)


,author,question_ending,question,preceding_turns,paragraph,paragraph_with_author_and_marker,full_turn,full_turn_with_marker,following_turns,subreddit,label
0,Person_B,why not,"Are we entitled, for example, to require that ...",Person_A: Alexander Hamilton once stressed the...,"Furthermore, if we are allowed to have additio...","Person_B: Furthermore, if we are allowed to ha...","Person_B: Sure, but what is that basis? Why th...","Person_B: Sure, but what is that basis? Why th...","Person_C: > &gt; Are we entitled, for example,...",changemyview,informational
1,Person_B,why,Why?,Person_A: Pretty much what it says in the titl...,&gt;and taxes are the natural cost of having a...,Person_B: &gt;and taxes are the natural cost o...,Person_B: &gt;I enjoy paying my taxes\n\nCan't...,Person_B: &gt;I enjoy paying my taxes\n\nCan't...,"Person_C: > &gt; Yes, but you're assuming gove...",changemyview,informational
2,Person_C,how,How?,Person_A: And how many people answered?\nPerso...,How? It is a matter of probability. Suppose a ...,Person_C: <ques>How?</ques> It is a matter of ...,"Person_C: If a country has a million people, a...","Person_C: If a country has a million people, a...",Person_A: > Of course if its based on random p...,europe,rhetorical
3,Person_B,how,And that changes things how?,"Person_A: The 2% obligation is not in place, y...",Person_B: And that changes things how? The US ...,Person_B: <ques>And that changes things how?</...,Person_B: And that changes things how? The US ...,Person_B: <ques>And that changes things how?</...,Person_C: > Let's take the world's most overbl...,europe,rhetorical
4,Person_B,why,Why?,Person_A: There is no conflict with the exampl...,Person_B: &gt;It's certainly possible. It's j...,Person_B: &gt;It's certainly possible. It's j...,Person_B: &gt;It's certainly possible. It's j...,Person_B: &gt;It's certainly possible. It's j...,Person_A: > Because where do you draw the line...,changemyview,informational


In [154]:
# converting the data to proper JSONL

train_split.to_json("../data/train_clean.jsonl", orient="records", lines=True, force_ascii=False)
dev_split.to_json("../data/dev_clean.jsonl", orient="records", lines=True, force_ascii=False)
test_split.to_json("../data/test_clean.jsonl", orient="records", lines=True, force_ascii=False)

train_split = pd.read_json("../data/train_clean.jsonl", lines=True)
dev_split = pd.read_json("../data/dev_clean.jsonl", lines=True)
test_split = pd.read_json("../data/test_clean.jsonl", lines=True)

I will use the preceding turn as my context condition for each question.

In [155]:
# empty preceding turns
print(train_split["preceding_turns"].isna().sum())

# preceding turns with empty string
print(train_split["preceding_turns"].str.strip().eq("").sum())

# non-textual content in preceding turns
print(train_split["preceding_turns"].str.contains(
    "<non-textual content>", regex=False
).sum())

# ... only non-textual content
print(train_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
).sum())

# checking for [removed] or [deleted]
print(train_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*(?:\[removed\]|\[deleted\])\s*"
).sum())

0
21
48
18
0


In [156]:
# empty preceding turns
print(dev_split["preceding_turns"].isna().sum())

# preceding turns with empty string
print(dev_split["preceding_turns"].str.strip().eq("").sum())

# non-textual content in preceding turns
print(dev_split["preceding_turns"].str.contains(
    "<non-textual content>", regex=False
).sum())

# ... only non-textual content
print(dev_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
).sum())

# checking for [removed] or [deleted]
print(dev_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*(?:\[removed\]|\[deleted\])\s*"
).sum())

0
10
8
3
0


In [157]:
# empty preceding turns
print(test_split["preceding_turns"].isna().sum())

# preceding turns with empty string
print(test_split["preceding_turns"].str.strip().eq("").sum())

# non-textual content in preceding turns
print(test_split["preceding_turns"].str.contains(
    "<non-textual content>", regex=False
).sum())

# ... only non-textual content
print(test_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
).sum())

# checking for [removed] or [deleted]
print(test_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*(?:\[removed\]|\[deleted\])\s*"
).sum())

0
27
51
17
0


Based on the results above, I will filter out preceding turns that:
- are empty strings (with or without whitespaces)
- are only non-textual content
  
This should exclude 96 examples in total.

In [158]:
# original label distribution

print("train:\n", train_split["label"].value_counts(normalize=True))
print("dev:\n", dev_split["label"].value_counts(normalize=True))
print("test:\n", test_split["label"].value_counts(normalize=True))

train:
 label
rhetorical       0.59375
informational    0.40625
Name: proportion, dtype: float64
dev:
 label
rhetorical       0.601942
informational    0.398058
Name: proportion, dtype: float64
test:
 label
rhetorical       0.659091
informational    0.340909
Name: proportion, dtype: float64


In [159]:
# filtering train

empty = train_split["preceding_turns"].str.strip().eq("")

non_textual = train_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
)

train_filtered = train_split[~(empty | non_textual)].copy()

train_filtered["label"].value_counts(normalize=True)

label
rhetorical       0.611594
informational    0.388406
Name: proportion, dtype: float64

In [160]:
# filtering dev

empty = dev_split["preceding_turns"].str.strip().eq("")

non_textual = dev_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
)

dev_filtered = dev_split[~(empty | non_textual)].copy()

dev_filtered["label"].value_counts(normalize=True)

label
rhetorical       0.644444
informational    0.355556
Name: proportion, dtype: float64

In [161]:
# filtering test

empty = test_split["preceding_turns"].str.strip().eq("")

non_textual = test_split["preceding_turns"].str.fullmatch(
    r"\s*Person_\w+:\s*<non-textual content>\s*"
)

test_filtered = test_split[~(empty | non_textual)].copy()

test_filtered["label"].value_counts(normalize=True)

label
rhetorical       0.681818
informational    0.318182
Name: proportion, dtype: float64

Filtering removed more informational questions than rhetorical questions.

In [162]:
# inspecting context lengths (by word counts)

train_filtered["context_length"] = (
    train_filtered["preceding_turns"].str.split().str.len()
)

print("train:\n", train_filtered["context_length"].describe())

dev_filtered["context_length"] = (
    dev_filtered["preceding_turns"].str.split().str.len()
)

print("dev:\n", dev_filtered["context_length"].describe())

test_filtered["context_length"] = (
    test_filtered["preceding_turns"].str.split().str.len()
)

print("test:\n", test_filtered["context_length"].describe())

train:
 count     345.000000
mean      366.110145
std       434.572076
min         4.000000
25%       121.000000
50%       228.000000
75%       459.000000
max      3361.000000
Name: context_length, dtype: float64
dev:
 count      90.000000
mean      342.411111
std       296.504956
min        11.000000
25%       117.250000
50%       250.500000
75%       468.000000
max      1606.000000
Name: context_length, dtype: float64
test:
 count     440.000000
mean      361.270455
std       466.644423
min         3.000000
25%       104.000000
50%       224.000000
75%       438.000000
max      4423.000000
Name: context_length, dtype: float64


Context length varies. Comparing mean and median, there seems to be a small number of longer context lengths dragging up the mean. Therefore, I will try to roughly length-match the contexts when creating the swapped condition.  

I will use quartiles to ensure roughly equal sized groups:  
shortest 25%  
next 25%  
next 25%  
longest 25%  
 

### Swapping the dev and test sets

In [163]:
# quartiles for dev

dev_filtered["length_group"] = pd.qcut(
    dev_filtered["context_length"],
    q=4,
    labels=False
)

dev_filtered["length_group"].value_counts()

length_group
0    23
3    23
2    22
1    22
Name: count, dtype: int64

In [164]:
# swapping group 0

group = dev_filtered[dev_filtered["length_group"] == 0]
indices = group.index.to_numpy()
#print(group)
print(indices)

# random shuffling indices with seed
rng = np.random.default_rng(42)
shuffled_indices = rng.permutation(indices)
print(shuffled_indices)

# rolling (shifting) indices one spot -> no self-swaps later
rolled_indices = np.roll(shuffled_indices, 1)
print(rolled_indices)

# swapping contexts
dev_filtered["swapped_context"] = None # new column
#dev_filtered.loc[shuffled_indices, "swapped_context"]
#dev_filtered.loc[rolled_indices, "preceding_turns"] # source contexts

dev_filtered.loc[shuffled_indices, "swapped_context"] = dev_filtered.loc[rolled_indices, "preceding_turns"].to_numpy()

[ 3  4  8 12 25 31 33 35 50 53 58 60 61 63 65 68 71 74 76 78 89 94 98]
[76 58 74 98 71 68 35 33 53 12  3 78 61 31 60 65 94  8 25 89  4 63 50]
[50 76 58 74 98 71 68 35 33 53 12  3 78 61 31 60 65 94  8 25 89  4 63]


In [165]:
# verifying

dev_filtered.loc[
    shuffled_indices[:3],
    ["question", "preceding_turns", "swapped_context"]
]

,question,preceding_turns,swapped_context
76,How so?,Person_A: <non-textual content>\nPerson_B: Eur...,Person_A: Yes but if you're not making profit....
58,Again how so?,Person_A: Not true.\nPerson_B: How so? Google ...,Person_A: <non-textual content>\nPerson_B: Eur...
74,"You can sue, but sue for what?",Person_A: I was sexually assaulted last year a...,Person_A: Not true.\nPerson_B: How so? Google ...


In [166]:
def swap_group(group_num: int, filtered_split, rng):

    # getting the indices
    group = filtered_split[filtered_split["length_group"] == group_num]
    indices = group.index.to_numpy()
    print("indices:", indices)

    # random shuffling indices with seed
    shuffled_indices = rng.permutation(indices)
    print("shuffled indices:", shuffled_indices)

    # rolling (shifting) indices one spot -> no self-swaps later
    rolled_indices = np.roll(shuffled_indices, 1)
    print("rolled indices:", rolled_indices)

    # swapping contexts in new column
    filtered_split.loc[shuffled_indices, "swapped_context"] = filtered_split.loc[rolled_indices, "preceding_turns"].to_numpy()

    return filtered_split


In [167]:
# creating random generator with seed

rng = np.random.default_rng(42)

In [168]:
# swapping within each of the four groups for the dev set

# initializing the new swapped context column
dev_filtered["swapped_context"] = None

for group_num in range(4):
    dev_filtered = swap_group(group_num, dev_filtered, rng)


indices: [ 3  4  8 12 25 31 33 35 50 53 58 60 61 63 65 68 71 74 76 78 89 94 98]
shuffled indices: [76 58 74 98 71 68 35 33 53 12  3 78 61 31 60 65 94  8 25 89  4 63 50]
rolled indices: [50 76 58 74 98 71 68 35 33 53 12  3 78 61 31 60 65 94  8 25 89  4 63]
indices: [  6  11  19  23  37  40  44  47  48  52  59  70  73  80  81  82  83  85
  86  92 100 101]
shuffled indices: [ 52 101  44  86  11  37  48  70  47  81  40 100  73   6  83  82  92  19
  59  23  80  85]
rolled indices: [ 85  52 101  44  86  11  37  48  70  47  81  40 100  73   6  83  82  92
  19  59  23  80]
indices: [  0   7   9  13  14  16  18  20  26  41  45  51  64  69  72  77  79  84
  93  96  97 102]
shuffled indices: [ 97  93  77  84  20  64  14  69  26  18  45   0  16   7  79  41   9  13
 102  72  96  51]
rolled indices: [ 51  97  93  77  84  20  64  14  69  26  18  45   0  16   7  79  41   9
  13 102  72  96]
indices: [10 15 17 21 22 24 27 28 29 34 36 39 42 43 46 54 66 75 87 88 91 95 99]
shuffled indices: [27 75 17 29 4

In [169]:
# checking for missing swapped contexts
dev_filtered["swapped_context"].isna().sum()

# checking for identical original and swapped contexts
(dev_filtered["preceding_turns"] == dev_filtered["swapped_context"]).sum()

np.int64(0)

In [172]:
# quartiles for test

test_filtered["length_group"] = pd.qcut(
    test_filtered["context_length"],
    q=4,
    labels=False
)

test_filtered["length_group"].value_counts()

length_group
0    112
3    110
2    110
1    108
Name: count, dtype: int64

In [173]:
# swapping within each of the four groups for the test set

# initializing the new swapped context column
test_filtered["swapped_context"] = None

for group_num in range(4):
    test_filtered = swap_group(group_num, test_filtered, rng)


indices: [  1   7   8  12  13  18  20  23  27  31  33  37  43  46  49  52  55  57
  61  69  72  73  77  82  83  84  89  94  96 100 125 128 135 138 145 147
 152 155 159 164 167 171 173 175 179 192 206 211 220 221 223 224 225 226
 235 237 240 243 246 252 258 260 261 263 264 269 276 282 288 290 291 293
 296 301 305 308 312 319 321 325 330 333 340 342 343 349 355 366 371 377
 378 381 383 390 393 407 411 426 429 438 441 442 443 446 447 462 466 467
 470 473 475 481]
shuffled indices: [145  89 441 206 159  23 325 481 167 240  72  20 467  83 171 426 296 155
 226 223 407 211 175 290  55 466  84 152 243  52 308 291 462 224  31 383
 128  82 319 411 264 138  12 100 125 282 442  27 235 343 340 429 377  94
 473 305 147 393   8 355 135  13  57 333 288 447 237 276 225  96  37  43
  49  73   1  33 371 261 381 330 164 269 258  69  46 260 438  18 349 252
 220 246 301 475 221   7 312  77 470 179 173 342 192 446 378 293  61 263
 321 366 390 443]
rolled indices: [443 145  89 441 206 159  23 325 481 167 240 

In [ ]:
# checking for missing swapped contexts
dev_filtered["swapped_context"].isna().sum()

# checking for identical original and swapped contexts
(dev_filtered["preceding_turns"] == dev_filtered["swapped_context"]).sum()

np.int64(0)